# 01 · Levels of context engineering — a code knowledge graph for qdrant

We climb three levels of context for a codebase agent, over a **configurable allowlist of qdrant crates**
(edit `CRATES` below to index as many or as few as you want):

| Level | What the agent gets | Ceiling |
|---|---|---|
| **L1** | grep / glob | can't resolve names or follow relationships |
| **L2** | an **index + search** (built here from deterministic extraction) | flat — can't tell apart two same-named types (`PostingList` exists in *two* crates), and **no edges** to answer "what depends on X?" |
| **L3** | a **context graph** (Neo4j) the agent **traverses** | — the payoff: name-collisions resolved + multi-hop blast radius |

The three *rungs* (syntactic → resolution → meaning) are *how* we build the index/graph; the *levels*
are the agent's capability. L2 and L3 share the same extracted rows — **L3 adds edges**.\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bcallender/agent-context-workshop/blob/main/notebooks/01_posting_list_context.ipynb)  ·  *Level 3 needs Neo4j — local Docker, or free Aura (see the next cell).*

In [15]:
# Colab bootstrap — no-op locally. In Colab: clone the repo, install it, hydrate the data.
import sys, os
if "google.colab" in sys.modules:
    import subprocess
    REPO = "/content/agent-context-workshop"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/bcallender/agent-context-workshop.git", REPO], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    subprocess.run(["bash", "scripts/setup_data.sh"], check=True)
    print("Colab ready — cloned, installed, data hydrated.")

In [16]:
# Level 3 needs a Neo4j. Local `docker compose up -d` is the default (no change needed).
# Using a remote instead — e.g. free Neo4j Aura (console.neo4j.io), which is how you'd run Level 3 in Colab?
# Uncomment and fill these three; without them, nb01 falls back to local and Level 3 just skips if none is up.
import os
# os.environ["NEO4J_URI"] = "neo4j+s://xxxxxxxx.databases.neo4j.io"
# os.environ["NEO4J_USER"] = "neo4j"
# os.environ["NEO4J_PASSWORD"] = "your-aura-password"

In [17]:
# Presentation mode (projection legibility). Harmless to run; skip/tweak freely. Browser zoom (Cmd +/-) also works.
from IPython.display import HTML, display
display(HTML("""<style>
.jp-RenderedHTMLCommon { font-size: 1.18rem; line-height: 1.65; }
.jp-RenderedHTMLCommon h1 { font-size: 2rem; }
.jp-RenderedHTMLCommon h2 { font-size: 1.5rem; }
.jp-RenderedText, .jp-OutputArea-output pre { font-size: 1.05rem; }
.cm-editor .cm-content { font-size: 1.05rem; }
</style>"""))

In [18]:
import logging
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

# Quiet the neo4j driver's server-notification logging: blast_radius's edge-type union includes
# CALLS (the deferred SCIP/Tier-C layer), which emits an "unknown relationship type" notification
# until those edges are loaded. Harmless; we keep the demo output clean.
logging.getLogger("neo4j").setLevel(logging.ERROR)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw_repos"
CACHE = ROOT / "data" / "cache"
# The indexing allowlist — add/remove crate names freely (rustdoc JSON is precomputed under data/cache).
# The PostingList collision needs posting_list + sparse; add "segment" for a ~5.7k-node graph.
CRATES = ["posting_list", "sparse", "gridstore", "quantization"]
DOC = CACHE / "cargo-target" / "doc"
CRATE_JSONS = {c: DOC / f"{c}.json" for c in CRATES}
PRIMARY = CRATES[0]                          # the crate we walk through Rung 0/1 in detail
CRATE = RAW / "qdrant" / "lib" / PRIMARY     # Rung 0 source (parsed statically)
RUSTDOC_JSON = CRATE_JSONS[PRIMARY]          # Rung 1 source (primary crate)

os.environ.setdefault("TQDM_DISABLE", "1")
load_dotenv(ROOT / ".env")  # no-op if absent; won't override already-exported vars
HAS_KEY = bool(os.getenv("OPENROUTER_API_KEY"))   # one key, any model (https://openrouter.ai/keys)

# Level 3 needs Neo4j. Smoke query (not a bare connect): print the server version so a wrong
# port / credentials / version fails loudly instead of silently disabling the graph half.
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "workshop123"))
HAS_NEO4J = False
try:
    from context_workshop.graph import connect
    _d = connect(NEO4J_URI, NEO4J_AUTH[0], NEO4J_AUTH[1])
    with _d.session() as _s:
        _ver = _s.run("CALL dbms.components() YIELD versions RETURN versions[0] AS v").single()["v"]
    _d.close()
    HAS_NEO4J = True
    print("neo4j:", _ver)
except Exception as _e:
    print("neo4j: unavailable —", str(_e)[:90], "(L3 cells will skip; run `docker compose up -d`)")

assert CRATE.exists(), "Run scripts/setup_data.sh first (hydrates the qdrant source + rustdoc JSON)."
_present = [c for c in CRATES if CRATE_JSONS[c].exists()]
print("indexing allowlist:", _present, "| primary (Rung 0/1 walkthrough):", PRIMARY, "| LLM key:", HAS_KEY)

neo4j: 5.26.27
indexing allowlist: ['posting_list', 'sparse', 'gridstore', 'quantization'] | primary (Rung 0/1 walkthrough): posting_list | LLM key: True


## The foundation (shared by L2 + L3): deterministic extraction

The index/graph is built from the code, statically — no LLM. **Rung 0** (tree-sitter) reads syntax;
**Rung 1** (rustdoc) resolves the program (re-exports → canonical def + `file:line`, effective visibility).

In [19]:
from context_workshop.parsers import extract_rust_crate, load_rust_rustdoc

ts = extract_rust_crate(CRATE)
print(f"Rung 0 (tree-sitter): {len(ts)} symbols")
rd = load_rust_rustdoc(RUSTDOC_JSON) if RUSTDOC_JSON.exists() else []
print(f"Rung 1 (rustdoc):     {len(rd)} symbols")
print("Rung 1 kinds:", dict(Counter(s.kind for s in rd)))

Rung 0 (tree-sitter): 112 symbols
Rung 1 (rustdoc):     85 symbols
Rung 1 kinds: {'reexport': 13, 'trait': 4, 'struct': 11, 'type_alias': 3, 'module': 1, 'const': 1, 'method': 52}


### The rung boundary, made concrete

`posting_list::PostingList` is the name a user writes (`use posting_list::PostingList`). Rung 0 sees the
`pub use` line but **can't resolve** where it's defined; Rung 1 resolves it to the canonical def + `file:line`.

In [20]:
def reexport(syms, name):
    return next((s for s in syms if s.is_reexport and s.name == name), None)

for label, syms in [("Rung 0 (tree-sitter)", ts), ("Rung 1 (rustdoc)", rd)]:
    s = reexport(syms, "PostingList")
    if s:
        print(f"{label:22} exported={s.exported_path}  canonical={s.canonical_path}  @ {s.filepath}:{s.line_start}")

Rung 0 (tree-sitter)   exported=posting_list::PostingList  canonical=None  @ /Users/brandoncallender/conductor/workspaces/agent-context-workshop/tianjin-v2/data/raw_repos/qdrant/lib/posting_list/src/lib.rs:48
Rung 1 (rustdoc)       exported=posting_list::PostingList  canonical=posting_list::posting_list::PostingList  @ lib/posting_list/src/posting_list.rs:27


## Level 2 — an index + search (across the whole allowlist)

The extracted rows *are* an index. Search it by name and it beats grep at name resolution. But across a
real multi-crate codebase two ceilings show up immediately:

1. **Collisions** — more than one crate defines a `PostingList`. The flat index lists both but can't tell
   you *which* one you mean.
2. **No edges** — index rows carry identity, location, and docs, but no relationship fields, so
   *"what depends on it?"* has nowhere to look.

Both are exactly what the graph (L3) fixes.

In [21]:
# Build the L2 index across the WHOLE allowlist (not just the primary crate).
rd_all = []
for _c, _jp in CRATE_JSONS.items():
    if _jp.exists():
        rd_all += load_rust_rustdoc(_jp)
print(f"flat index: {len(rd_all)} symbols across {len(_present)} crates")

# Search by name and hit the COLLISION: more than one type is named PostingList.
plist = [s for s in rd_all if s.name == "PostingList"]
print(f"\nname == 'PostingList' -> {len(plist)} distinct types (a collision the flat index can't resolve):")
for s in plist:
    print(f"  {s.kind:8} {s.qualified_name}  ({s.filepath}:{s.line_start})")

# And the rows carry no relationship fields to answer "what depends on each?"
row = rd_all[0]
print("\nrelationship fields on an index row:",
      {k: getattr(row, k) for k in ("signature", "parameters", "bases")})
print("-> a flat index can't disambiguate the collision by intent, and has no edges to traverse. That's L3.")

flat index: 816 symbols across 4 crates

name == 'PostingList' -> 3 distinct types (a collision the flat index can't resolve):
  reexport posting_list::PostingList  (lib/posting_list/src/posting_list.rs:27)
  struct   posting_list::posting_list::PostingList  (lib/posting_list/src/posting_list.rs:27)
  struct   sparse::index::posting_list::PostingList  (lib/sparse/src/index/posting_list.rs:12)

relationship fields on an index row: {'signature': None, 'parameters': None, 'bases': None}
-> a flat index can't disambiguate the collision by intent, and has no edges to traverse. That's L3.


## Level 3 — the context graph

Same extracted rows, now with **edges**. `build_rust_graph` derives typed relationships
deterministically from the resolved rustdoc — containment, methods, trait impls, re-exports, and
signature/field type-uses — then we load nodes + edges into Neo4j and **traverse**.

In [22]:
from context_workshop.graph import build_rust_graph, dedupe_edges

nodes, edges = [], []
for _c, _jp in CRATE_JSONS.items():
    if not _jp.exists():
        print(f"  (skip {_c}: no rustdoc json)"); continue
    _n, _e = build_rust_graph(_jp)
    nodes += _n; edges += _e

# Restrict to the allowlisted crates' own symbols. Foreign-impl methods (e.g. a method a crate adds to
# half::f16 or alloc::Vec) are keyed to the *external* crate and have no parent node here — keeping them
# would leave orphan nodes. Dropping them makes "indexing allowlist" literal; cross-crate edges *within*
# the allowlist are kept.
_allow = set(CRATES)
nodes = list({nd["qualified_name"]: nd for nd in nodes
              if nd["qualified_name"].split("::", 1)[0] in _allow}.values())
_keep = {nd["qualified_name"] for nd in nodes}
edges = [e for e in dedupe_edges(edges) if e.src in _keep and e.dst in _keep]
print(f"{len(nodes)} nodes, {len(edges)} edges across {len(_present)} crates")
print("edge types:", dict(Counter(e.type for e in edges)))

730 nodes, 971 edges across 4 crates
edge types: {'CONTAINS': 176, 'REEXPORTS': 56, 'HAS_METHOD': 522, 'IMPLEMENTS': 14, 'RETURNS': 136, 'TAKES': 49, 'HAS_FIELD': 18}


In [23]:
if HAS_NEO4J:
    from context_workshop.graph import connect, load, GraphTools
    driver = connect(NEO4J_URI, NEO4J_AUTH[0], NEO4J_AUTH[1])
    stats = load(driver, nodes, edges, reset=True)   # raises if any node is isolated (key drift)
    with driver.session() as s:
        n = s.run("MATCH (n:Symbol) RETURN count(n) AS n").single()["n"]
    assert n == stats["nodes"], f"node-count mismatch after load: {n} vs {stats['nodes']}"
    gt = GraphTools(driver)
    print("loaded:", stats, "| node-count check OK:", n)
else:
    driver = gt = None
    print("Neo4j not available — skipping L3 load (run `docker compose up -d`).")

loaded: {'nodes': 730, 'edges': 971, 'isolated': 0} | node-count check OK: 730


### The payoff — blast radius (what L1/L2 structurally can't do)

`blast_radius` walks *inbound* dependency edges (returns/takes/has-field/has-method/implements) to find
everything that breaks if the target changes — multi-hop, with `file:line` evidence on every hit.

No wrapper, no magic — `blast_radius` is **one Cypher query**, written out in the cell below.

In [25]:
if HAS_NEO4J:
    # blast_radius is ONE Cypher query — nothing hidden behind a wrapper. From the target symbol,
    # walk dependency edges INBOUND (returns / takes / has-field / has-method / implements) up to
    # `depth` hops, and collect every symbol that reaches it — with the edge-path and file:line on
    # each hit. (This is exactly what the agent's `blast_radius` tool runs. The tool also unions in
    # CALLS — the deferred SCIP call-graph layer — which is empty until you load those edges, so the
    # result here is identical.)
    def blast_radius(qualified_name, depth=2):
        cypher = f"""
            MATCH (target:Symbol {{qualified_name: $qn}})
                  <-[rels:RETURNS|TAKES|HAS_FIELD|HAS_METHOD|IMPLEMENTS*1..{depth}]-(dependent:Symbol)
            RETURN DISTINCT dependent.qualified_name AS qualified_name,
                            dependent.filepath       AS filepath,
                            dependent.line_start      AS line_start,
                            [r IN rels | type(r)]     AS path
        """
        with gt._driver.session() as session:
            return session.run(cypher, qn=qualified_name).data()

    TARGET = "posting_list::posting_list::PostingList"
    deps = blast_radius(TARGET, depth=2)
    print(f"blast_radius({TARGET}, depth=2) -> {len(deps)} dependents")
    for d in deps[:10]:
        print(f"  {d['qualified_name']}  ({d['filepath']}:{d['line_start']})  via {'->'.join(d['path'])}")

    # GOLDEN PROBE — the demo-critical invariant; the headless run FAILS here if the payoff is empty.
    assert deps, "blast_radius returned no dependents — graph load or edges are broken"
    assert all(d["filepath"] and d["line_start"] is not None for d in deps), "every dependent needs file:line"
    print("\ngolden probe OK")
else:
    print("(skipped — needs Neo4j)")


blast_radius(posting_list::posting_list::PostingList, depth=2) -> 6 dependents
  posting_list::view::PostingListView::to_owned  (lib/posting_list/src/view.rs:49)  via RETURNS
  posting_list::view::PostingListView  (lib/posting_list/src/view.rs:16)  via RETURNS->HAS_METHOD
  posting_list::posting_list::PostingList::clone  (lib/posting_list/src/posting_list.rs:26)  via RETURNS
  posting_list::posting_list::PostingList  (lib/posting_list/src/posting_list.rs:27)  via RETURNS->HAS_METHOD
  posting_list::builder::PostingBuilder::build  (lib/posting_list/src/builder.rs:36)  via RETURNS
  posting_list::builder::PostingBuilder  (lib/posting_list/src/builder.rs:11)  via RETURNS->HAS_METHOD

golden probe OK


### The collision, resolved

L2 found two `PostingList` types but couldn't tell them apart by intent. The graph keeps them as distinct
nodes (by canonical path) and gives each its **own** blast radius — the disambiguation flat search couldn't.

In [26]:
if HAS_NEO4J:
    # the two *struct definitions* named PostingList (skip the re-export alias, which isn't a node)
    for qn in sorted({s.qualified_name for s in rd_all if s.name == "PostingList" and s.kind == "struct"}):
        deps = blast_radius(qn, depth=2)
        print(f"{qn}  ->  {len(deps)} dependents")
        for d in deps[:4]:
            print(f"    {d['qualified_name']}  ({d['filepath']}:{d['line_start']})")
else:
    print("(skipped — needs Neo4j)")

posting_list::posting_list::PostingList  ->  6 dependents
    posting_list::view::PostingListView::to_owned  (lib/posting_list/src/view.rs:49)
    posting_list::view::PostingListView  (lib/posting_list/src/view.rs:16)
    posting_list::posting_list::PostingList::clone  (lib/posting_list/src/posting_list.rs:26)
    posting_list::posting_list::PostingList  (lib/posting_list/src/posting_list.rs:27)
sparse::index::posting_list::PostingList  ->  14 dependents
    sparse::index::posting_list::PostingList::new_one  (lib/sparse/src/index/posting_list.rs:28)
    sparse::index::posting_list::PostingList  (lib/sparse/src/index/posting_list.rs:12)
    sparse::index::posting_list::PostingBuilder::build  (lib/sparse/src/index/posting_list.rs:140)
    sparse::index::posting_list::PostingBuilder  (lib/sparse/src/index/posting_list.rs:117)


## Level 3, composed by an agent

A minimal in-notebook Pydantic AI agent gets the graph traversal tools and composes them to answer a
question — citing the symbols it relied on. The model is a labeled knob (swap freely).

In [27]:
def dump_trace(messages):
    """Pair up the agent's tool calls with their results/errors from the message history."""
    for m in messages:
        for part in getattr(m, "parts", []):
            kind = type(part).__name__
            if kind == "ToolCallPart":
                print(f"  -> {part.tool_name}({part.args})")
            elif kind == "ToolReturnPart":
                print(f"  <- {part.tool_name}: {str(part.content)[:120]}")
            elif kind == "RetryPromptPart":
                print(f"  x {getattr(part, 'tool_name', None)} retry: {str(getattr(part, 'content', ''))[:160]}")

if HAS_NEO4J and HAS_KEY:
    from context_workshop.graph import build_graph_agent
    AGENT_MODEL = os.getenv("AGENT_MODEL", "openrouter:openai/gpt-5.4-mini")   # one OPENROUTER_API_KEY, any model (e.g. "openrouter:anthropic/claude-sonnet-4")
    agent = build_graph_agent(AGENT_MODEL, gt)
    # Natural question, bare name: a tool-calling-strong model resolves it via `search` first,
    # then traverses with blast_radius — the agent composes the context layer itself.
    res = await agent.run(
        "What breaks if I change `PostingList`? List each dependent symbol with its file:line.")
    print(res.output)
    print("\n--- tool trace ---")
    dump_trace(res.all_messages())
else:
    print("Need Neo4j + an LLM key to run the agent.")

Here are the dependent symbols the graph reports for `posting_list::posting_list::PostingList`:

- `posting_list::view::PostingListView::to_owned` — `lib/posting_list/src/view.rs:49`
- `posting_list::view::PostingListView` — `lib/posting_list/src/view.rs:16`
- `posting_list::posting_list::PostingList::view` — `lib/posting_list/src/posting_list.rs:88`
- `posting_list::view::PostingListView::clone` — `lib/posting_list/src/view.rs:15`
- `posting_list::posting_list::PostingList::clone` — `lib/posting_list/src/posting_list.rs:26`
- `posting_list::posting_list::PostingList` — `lib/posting_list/src/posting_list.rs:27`
- `posting_list::builder::PostingBuilder::build` — `lib/posting_list/src/builder.rs:36`
- `posting_list::builder::PostingBuilder` — `lib/posting_list/src/builder.rs:11`
- `posting_list::posting_list::PostingList::from` — `lib/posting_list/src/builder.rs:123`

Notes:
- I used `posting_list::posting_list::PostingList` as the resolved symbol for `PostingList`.
- These are the symbo

## Your turn — extend the agent with a tool you write in Cypher

The context layer is extensible: write a Cypher query, wrap it as a tool, and the agent can compose it.
The curated tools traverse *relationships*; here we add a capability they don't have — **"what symbols
are defined in a given file?"** — purely from one Cypher query.

**Direct use** — `cypher_tool(gt, "<cypher>")` returns a callable you invoke with keyword params.
**Agent use** — give the agent a *typed* function (so it knows the parameter), via `extra_tools=[...]`.
Edit the query to answer your own question.

In [13]:
if HAS_NEO4J:
    from context_workshop.graph import cypher_tool

    # Direct-use: wrap any Cypher as a callable tool (the build-your-own primitive).
    symbols_in_file_q = cypher_tool(gt,
        "MATCH (n:Symbol) WHERE n.filepath CONTAINS $path "
        "RETURN n.qualified_name AS qualified_name, n.filepath AS filepath, n.line_start AS line_start "
        "ORDER BY n.line_start")
    print("symbols defined in builder.rs (direct cypher_tool call):")
    for s in symbols_in_file_q(path="builder.rs"):
        print(f"  {s['qualified_name']}  ({s['filepath']}:{s['line_start']})")
else:
    print("(skipped — needs Neo4j)")

symbols defined in builder.rs (direct cypher_tool call):
  sparse::index::inverted_index::inverted_index_ram_builder  (lib/sparse/src/index/inverted_index/inverted_index_ram_builder.rs:1)
  posting_list::builder::PostingBuilder  (lib/posting_list/src/builder.rs:11)
  sparse::index::inverted_index::inverted_index_ram_builder::InvertedIndexBuilder  (lib/sparse/src/index/inverted_index/inverted_index_ram_builder.rs:12)
  posting_list::builder::PostingBuilder::default  (lib/posting_list/src/builder.rs:16)
  sparse::index::inverted_index::inverted_index_ram_builder::InvertedIndexBuilder::default  (lib/sparse/src/index/inverted_index/inverted_index_ram_builder.rs:19)
  posting_list::builder::PostingBuilder::new  (lib/posting_list/src/builder.rs:24)
  sparse::index::inverted_index::inverted_index_ram_builder::InvertedIndexBuilder::new  (lib/sparse/src/index/inverted_index/inverted_index_ram_builder.rs:25)
  posting_list::builder::PostingBuilder::add  (lib/posting_list/src/builder.rs:28)
  spa

In [28]:
if HAS_NEO4J and HAS_KEY:
    # Agent-use: a TYPED wrapper so pydantic-ai can register it and the model can pass `path`.
    # (Not a curated tool name — build_graph_agent already registers search/blast_radius/methods_of/...)
    def symbols_in_file(path: str) -> list[dict]:
        """Symbols defined in a source file, matched by path substring, with file:line."""
        cy = ("MATCH (n:Symbol) WHERE n.filepath CONTAINS $path "
              "RETURN n.qualified_name AS qualified_name, n.filepath AS filepath, n.line_start AS line_start "
              "ORDER BY n.line_start")
        with gt._driver.session() as s:
            return s.run(cy, path=path).data()

    agent2 = build_graph_agent(AGENT_MODEL, gt, extra_tools=[symbols_in_file])
    res2 = await agent2.run("What symbols are defined in builder.rs? List them with file:line.")
    print(res2.output)
    print("\n--- tool trace ---")
    dump_trace(res2.all_messages())
else:
    print("Need Neo4j + an LLM key to run the agent with your custom tool.")

Symbols defined in `builder.rs`:

- `posting_list::builder::PostingBuilder` — `lib/posting_list/src/builder.rs:11`
- `posting_list::builder::PostingBuilder::default` — `lib/posting_list/src/builder.rs:16`
- `posting_list::builder::PostingBuilder::new` — `lib/posting_list/src/builder.rs:24`
- `posting_list::builder::PostingBuilder::add` — `lib/posting_list/src/builder.rs:28`
- `posting_list::builder::PostingBuilder::build` — `lib/posting_list/src/builder.rs:36`
- `posting_list::builder::PostingBuilder::add_id` — `lib/posting_list/src/builder.rs:117`
- `posting_list::posting_list::PostingList::from` — `lib/posting_list/src/builder.rs:123`

Also shown in the same file listing:

- `sparse::index::inverted_index::inverted_index_ram_builder` — `lib/sparse/src/index/inverted_index/inverted_index_ram_builder.rs:1`
- `sparse::index::inverted_index::inverted_index_ram_builder::InvertedIndexBuilder` — `lib/sparse/src/index/inverted_index/inverted_index_ram_builder.rs:12`
- `sparse::index::inverte

### Don't know Cypher? Describe what you want — the agent writes it

The graph has a *described schema*, so an LLM can translate natural language into Cypher. Pick an example
request (or write your own); given the schema, the agent generates a **read-only** query, which we wrap as
a tool and run. This is the schema-as-context payoff: you don't need the query language, just what you want.

In [30]:
SCHEMA = """
Nodes: every node has label :Symbol plus a kind label
  (:Module :Struct :Enum :Trait :TypeAlias :Function :Method :Const :Crate).
  Key properties: qualified_name (unique canonical ::-path), name, kind, crate,
  filepath, line_start, docstring, is_public, effective_public.
  :Public marks the effective public-API surface.
Edges (direction matters):
  (:Module)-[:CONTAINS]->(:Symbol)              module contains an item
  (:Struct|:Enum|:Trait)-[:HAS_METHOD]->(:Method)
  (:Struct|:Enum)-[:IMPLEMENTS]->(:Trait)       (synthetic=true for derive impls)
  (:Function|:Method)-[:RETURNS]->(:Symbol)     return type
  (:Function|:Method)-[:TAKES]->(:Symbol)       parameter type
  (:Struct|:Enum)-[:HAS_FIELD]->(:Symbol)       field type
  (:Module)-[:REEXPORTS]->(:Symbol)             pub use -> canonical def
Blast radius = inbound dependency edges to a target symbol.
"""

if HAS_NEO4J and HAS_KEY:
    import re
    from pydantic_ai import Agent

    _WRITE = re.compile(r"\b(CREATE|MERGE|SET|DELETE|REMOVE|DROP)\b", re.I)
    _cypher_writer = Agent(AGENT_MODEL, system_prompt=(
        "You translate a request into ONE read-only Neo4j Cypher query for the given schema. "
        "Return ONLY the query — no prose, no markdown fences. Prefer returning qualified_name, "
        "filepath, line_start for matched symbols, and add a LIMIT 25."))

    async def nl_to_cypher(want: str) -> str:
        out = (await _cypher_writer.run(f"Request: {want}\n\nSchema:\n{SCHEMA}")).output.strip()
        out = out.removeprefix("```cypher").removeprefix("```").removesuffix("```").strip()
        if _WRITE.search(out):
            raise ValueError(f"refusing non-read-only Cypher:\n{out}")
        return out
    print("nl_to_cypher ready — describe what you want, get Cypher.")
else:
    print("Need Neo4j + an LLM key for NL->Cypher.")

nl_to_cypher ready — describe what you want, get Cypher.


In [35]:
WANTS = [
    "every struct that implements a trait, with the trait it implements",
    "functions or methods that take a PostingList as a parameter",
    "all public traits in the crate",
]
WANT = WANTS[0]   # <- pick another index, or replace with your own sentence

if HAS_NEO4J and HAS_KEY:
    from context_workshop.graph import cypher_tool
    cy = await nl_to_cypher(WANT)
    print("WANT:", WANT)
    print("generated Cypher:\n", cy, "\n")
    try:
        rows = cypher_tool(gt, cy)()
        print(f"{len(rows)} rows:")
        for r in rows[:10]:
            print("  ", r)
    except Exception as e:
        print("query failed — refine the request and re-run:", str(e)[:200])
else:
    print("(skipped — needs Neo4j + key)")

WANT: every struct that implements a trait, with the trait it implements
generated Cypher:
 MATCH (s:Struct)-[:IMPLEMENTS]->(t:Trait)
RETURN s.qualified_name AS struct_qualified_name, s.filepath AS struct_filepath, s.line_start AS struct_line_start, t.qualified_name AS trait_qualified_name, t.filepath AS trait_filepath, t.line_start AS trait_line_start
LIMIT 25 

14 rows:
   {'struct_qualified_name': 'posting_list::value_handler::UnsizedHandler', 'struct_filepath': 'lib/posting_list/src/value_handler.rs', 'struct_line_start': 76, 'trait_qualified_name': 'posting_list::value_handler::ValueHandler', 'trait_filepath': 'lib/posting_list/src/value_handler.rs', 'trait_line_start': 33}
   {'struct_qualified_name': 'posting_list::value_handler::SizedHandler', 'struct_filepath': 'lib/posting_list/src/value_handler.rs', 'struct_line_start': 56, 'trait_qualified_name': 'posting_list::value_handler::ValueHandler', 'trait_filepath': 'lib/posting_list/src/value_handler.rs', 'trait_line_start': 33}
 

## A taste of Rung 2 — meaning (the "+enhanced search" upgrade, previewed)

So far everything is deterministic (structure + resolution). The third rung is **meaning**: an LLM
summarizes each symbol and embeds it. That's where **Fenic** (the transformation layer) earns its keep —
and where *semantic* search comes from. Here's a one-cell taste on a handful of nodes; the full version
(summaries + embeddings as node properties + a Neo4j vector index + a semantic `search` tool) is the next pass.

In [36]:
if HAS_KEY and rd:
    import fenic as fc
    from context_workshop.parsers.schema import to_arrow
    (CACHE / "fenic").mkdir(parents=True, exist_ok=True)

    sem = fc.SemanticConfig(
        language_models={"mini": fc.OpenRouterLanguageModel(model_name="openai/gpt-4o-mini", rpm=500, tpm=200_000)},
        default_language_model="mini")  # the index-building model — a separate knob from the agent model
    sem_session = fc.Session.get_or_create(
        fc.SessionConfig(app_name="nb01_rung2_preview", semantic=sem, db_path=str(CACHE / "fenic")))
    sample = [s for s in rd if s.kind in ("struct", "trait", "function")][:5]
    preview = (sem_session.create_dataframe(to_arrow(sample))
        .with_column("blurb", fc.text.jinja("Rust {{k}} `{{qn}}`\n{{d}}", strict=False,
            k=fc.col("kind"), qn=fc.col("qualified_name"), d=fc.col("docstring")))
        .with_column("summary", fc.semantic.map(
            "In one concise sentence, what is this Rust item for?\n\n{{b}}", b=fc.col("blurb"))))
    preview.select("kind", "name", "summary").show()
else:
    print("Skipping the Rung-2 preview (needs an LLM key + Rung-1 rows).")

┌────────┬─────────────────┬───────────────────────────────────────────────────────────────────────┐
│ kind   ┆ name            ┆ summary                                                               │
╞════════╪═════════════════╪═══════════════════════════════════════════════════════════════════════╡
│ trait  ┆ ValueHandler    ┆ The `posting_list::value_handler::ValueHandler` trait abstracts the   │
│        ┆                 ┆ handling of fixed-size and variable-size values in a PostingList,     │
│        ┆                 ┆ enabling a unified implementation of `from_builder`.                  │
│ struct ┆ PostingVisitor  ┆ The `posting_list::visitor::PostingVisitor` struct is a visitor that  │
│        ┆                 ┆ caches the most recent decompressed chunk of IDs from a posting list. │
│ struct ┆ PostingIterator ┆ The `posting_list::iterator::PostingIterator` struct in Rust is used  │
│        ┆                 ┆ to iterate over a posting list in a way that efficiently proce

## Where this goes next

- **+ semantic search** — load the Rung-2 `summary`/`embedding` onto the graph nodes, add a Neo4j
  **vector index**, and a semantic `search` tool. That upgrades L2/L3 search from lexical to meaning-aware.
- **Does L3 actually beat L1?** — the honest A/B eval: this graph agent vs a real grep/glob agent, scored
  on dependency-set accuracy and efficiency, at scale (notebooks 02 / 03).

The levels: **L1 grep → L2 index+search → L3 context graph.** You just built the climb.